# [TARGET] Skillnox AI — Fine-Tune Qwen3-8B (High-Performance A100/L4 Config)

**Model**: Qwen3-8B (text-only)  
**Method**: QLoRA (4-bit) via Unsloth  
**Dataset**: 50,000 examples (Full Dataset)  
**Hardware**: NVIDIA A100 (40GB/80GB), L4 (24GB), or RTX 3090/4090 (24GB)  
**Estimated Time**: ~30-45 minutes (A100) or ~1.5 hours (L4)  

## Optimized Production Settings
- [OK] **Full 50k Dataset** — Set to train on all examples
- [OK] **Dataset Packing Enabled** — Groups short examples into dense blocks for 3-4x speedup
- [OK] **Higher LoRA Capacity (Rank 32)** — Captures deeper text relationships for complex answers
- [OK] **BF16 Precision Enabled** — Native Ampere hardware acceleration (avoids gradient overflow)
- [OK] **Larger Batches** — Batch size 8/16 per device to fully saturate GPU cores

---
## Cell 1 — Install Dependencies & GPU Check

In [ ]:
%%time
!pip install -q -U unsloth transformers datasets peft accelerate bitsandbytes trl safetensors huggingface_hub matplotlib
!pip uninstall -y hf-transfer

import torch
import os

print("\n" + "="*60)
print("HIGH-PERFORMANCE ENVIRONMENT CHECK")
print("="*60)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB")
    print(f"Supports BF16: {torch.cuda.is_bf16_supported()}")
else:
    print("[WARN] NO GPU DETECTED!")

print(f"\nSystem CPU RAM: {os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / (1024**3):.1f} GB")
print("=== Environment ready ===")

---
## Cell 2 — High-Performance Configuration Dashboard

In [ ]:
# =============================================================
# PRODUCTION CONFIGURATION — Optimized for A100 / L4 / RTX 4090
# =============================================================

# Model
MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"  # Quantized Qwen3-8B base model
MAX_SEQ_LEN = 1024                          # Sequence context length

# LoRA Configuration
LORA_RANK = 32             # LoRA rank 32 for complex learning capacity
LORA_ALPHA = 64            # 2x rank
LORA_DROPOUT = 0.05

# Dataset Scale
MAX_EXAMPLES = None        # Set to None to use ALL 50,000 examples
VALIDATION_SPLIT = 0.05    # 5% held out for evaluation

# Hardware Optimization
BATCH_SIZE = 8             # High VRAM batch size (8 for A10G/L4/RTX4090, 16 for A100)
GRAD_ACCUM_STEPS = 4       # Effective batch size = 8 x 4 = 32
USE_BF16 = True            # True for Ampere GPUs (A100/L4/RTX3090/4090) - faster & more stable
PACKING = True             # True to pack multiple short samples into sequences (3-4x speedup)

# Training Hyperparameters
NUM_EPOCHS = 5             # 5 epochs for comprehensive fine-tuning
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

# Checkpointing & Logs
SAVE_STEPS = 250           # Save checkpoint every N steps
SAVE_TOTAL_LIMIT = 3       # Keep last 3 checkpoints
EVAL_STEPS = 250           # Evaluate loss every N steps
LOGGING_STEPS = 10         # Log metrics frequently

# Paths
OUTPUT_DIR = "./output"
LORA_DIR = "./lora_adapter"
GGUF_DIR = "./skillnox-gguf"

print("="*60)
print("A100/L4 TRAINING CONFIGURATION")
print("="*60)
print(f"Model Name:     {MODEL_NAME}")
print(f"LoRA Capacity:  Rank {LORA_RANK} (Alpha: {LORA_ALPHA})")
print(f"Dataset Size:   {'ALL 50,000' if MAX_EXAMPLES is None else MAX_EXAMPLES} examples")
print(f"Precision:      {'BF16 (Ampere accelerated)' if USE_BF16 else 'FP16'}")
print(f"Packing:        {PACKING} (Speed optimized)")
print(f"Batch Size:     {BATCH_SIZE} x {GRAD_ACCUM_STEPS} = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Training Epochs:{NUM_EPOCHS}")
print("="*60)

---
## Cell 3 — Load & Prepare Dataset

In [ ]:
%%time
import json
import random
from datasets import Dataset

# Path to the uploaded dataset file (update this path based on where you upload the file)
DATA_PATH = "extended_training_data.jsonl"

if not os.path.exists(DATA_PATH):
    # Attempt to locate in parent directories or search
    for root, dirs, files in os.walk("."):
        for f in files:
            if f.endswith("extended_training_data.jsonl"):
                DATA_PATH = os.path.join(root, f)
                break

print(f"Loading dataset: {DATA_PATH}")

# Load raw dataset
raw_data = []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        raw_data.append(json.loads(line))

print(f"Loaded {len(raw_data):,} raw training examples")

# Shuffle and cap if requested
random.seed(42)
random.shuffle(raw_data)
if MAX_EXAMPLES and MAX_EXAMPLES < len(raw_data):
    raw_data = raw_data[:MAX_EXAMPLES]
    print(f"Capped to {MAX_EXAMPLES:,} examples")

SYSTEM_PROMPT = (
    "You are SkillnoxAI, an expert AI-powered interview preparation and placement assistant. "
    "Your capabilities include: Interview Question Generation, Answer Evaluation (score 0-100), "
    "Resume Analysis, Communication Assessment, Group Discussion Evaluation, and Aptitude Test Evaluation. "
    "Be STRICT but constructive. Do NOT inflate scores. "
    "For evaluations, always use: Score: [number]\nFeedback: [text]. "
    "Focus on Indian campus placement context (TCS, Infosys, Wipro, Accenture, etc.)."
)

def format_example(example):
    instruction = example["instruction"]
    inp = json.dumps(example["input"]) if isinstance(example["input"], dict) else str(example["input"])
    out = json.dumps(example["output"]) if isinstance(example["output"], dict) else str(example["output"])
    
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{instruction}\n{inp}<|im_end|>\n"
        f"<|im_start|>assistant\n{out}<|im_end|>"
    )
    return {"text": text}

formatted = [format_example(ex) for ex in raw_data]

# Split into train/eval
split_idx = int(len(formatted) * (1 - VALIDATION_SPLIT))
train_formatted = formatted[:split_idx]
eval_formatted = formatted[split_idx:]

train_dataset = Dataset.from_list(train_formatted)
eval_dataset = Dataset.from_list(eval_formatted)

# Clean memory immediately to free up system RAM
import gc
del raw_data
del formatted
del train_formatted
del eval_formatted
gc.collect()

print(f"\nTrain size: {len(train_dataset):,}")
print(f"Eval size:  {len(eval_dataset):,}")
print("System memory cleared successfully!")

---
## Cell 4 — Load Qwen3-8B with FastLanguageModel

In [ ]:
%%time
import os
import shutil
from pathlib import Path
from huggingface_hub.constants import HF_HUB_CACHE

# Dynamically locate and clean corrupted HuggingFace cache directory (handles custom HF_HOME/HF_HUB_CACHE env variables)
model_cache_name = f"models--{MODEL_NAME.replace('/', '--')}"
cache_dir = Path(HF_HUB_CACHE) / model_cache_name
if cache_dir.exists():
    print(f"[CLEANUP] Cleaning corrupted cache directory: {cache_dir}")
    shutil.rmtree(cache_dir)
else:
    print(f"[CLEANUP] Cache directory not found at: {cache_dir} (starting fresh)")

# Disable hf_transfer to use stable python requests download (fixes connection drops/SSL errors on cloud VMs)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

from unsloth import FastLanguageModel
from huggingface_hub import login

# Log in to Hugging Face
import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    login(token=HF_TOKEN)

print(f"Loading {MODEL_NAME} on GPU...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # Auto-detect hardware precision
    token=HF_TOKEN if HF_TOKEN else None,
)

# Set up LoRA adapters with Rank 32 for enhanced training
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",  # Minimizes VRAM footprints
    random_state=42,
)

print("\n=== Model compiled with LoRA successfully ===")
model.print_trainable_parameters()

if torch.cuda.is_available():
    print(f"VRAM allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB")


---
## Cell 5 — Train with Checkpoints & Resume Support

In [ ]:
%%time
import glob
import time
import traceback

# Auto-detect existing checkpoints
resume_checkpoint = None
checkpoints = sorted(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
if checkpoints:
    resume_checkpoint = checkpoints[-1]
    print(f"[RESUME] Resuming training from: {resume_checkpoint}")
else:
    print("[NEW] Starting a fresh training run")

# Create high-performance trainer
try:
    from trl import SFTTrainer, SFTConfig
    print("Configuring SFTConfig (TRL >= 0.14)")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=SFTConfig(
            output_dir=OUTPUT_DIR,
            dataset_text_field="text",
            max_seq_length=MAX_SEQ_LEN,
            packing=PACKING,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            bf16=USE_BF16,
            fp16=not USE_BF16,
            logging_steps=LOGGING_STEPS,
            save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            optim="adamw_8bit",
            seed=42,
            report_to="none",
        ),
    )
except ImportError:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    print("Configuring TrainingArguments (older TRL)")
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        packing=PACKING,
        args=TrainingArguments(
            output_dir=OUTPUT_DIR,
            num_train_epochs=NUM_EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM_STEPS,
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_ratio=WARMUP_RATIO,
            weight_decay=WEIGHT_DECAY,
            bf16=USE_BF16,
            fp16=not USE_BF16,
            logging_steps=LOGGING_STEPS,
            save_steps=SAVE_STEPS,
            save_total_limit=SAVE_TOTAL_LIMIT,
            eval_strategy="steps",
            eval_steps=EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            optim="adamw_8bit",
            seed=42,
            report_to="none",
        ),
    )

print("\n[GO] Starting fine-tuning loop...")
start_time = time.time()
stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
elapsed = time.time() - start_time

print(f"\n[OK] Training Complete!")
print(f"Final loss:     {stats.training_loss:.4f}")
print(f"Training time:  {elapsed/3600:.2f} hours")

# Save LoRA adapter
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"[SAVE] LoRA adapter saved to {LORA_DIR}")

---
## Cell 6 — Loss Plots

In [ ]:
import matplotlib.pyplot as plt

if "trainer" in dir() and trainer is not None:
    train_losses = []
    eval_losses = []
    train_steps = []
    eval_steps_log = []

    for log in trainer.state.log_history:
        if "loss" in log and "eval_loss" not in log:
            train_losses.append(log["loss"])
            train_steps.append(log["step"])
        if "eval_loss" in log:
            eval_losses.append(log["eval_loss"])
            eval_steps_log.append(log["step"])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    ax1.plot(train_steps, train_losses, "b-", alpha=0.7, linewidth=1.5)
    ax1.set_xlabel("Steps")
    ax1.set_ylabel("Training Loss")
    ax1.set_title("Training Loss")
    ax1.grid(True, alpha=0.3)

    if eval_losses:
        ax2.plot(eval_steps_log, eval_losses, "r-o", markersize=4)
        ax2.set_xlabel("Steps")
        ax2.set_ylabel("Eval Loss")
        ax2.set_title("Validation Loss")
        ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_loss.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Trainer data unavailable for plotting")

---
## Cell 7 — Model Verification

In [ ]:
FastLanguageModel.for_inference(model)

def test_model(system, user_msg, max_tokens=300):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_msg},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to("cuda")
    out = model.generate(inputs, max_new_tokens=max_tokens, temperature=0.7, do_sample=True)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

SYS = "You are SkillnoxAI, an expert interview preparation and placement assistant."

TESTS = [
    ("Technical Question Generation",
     '{"question_type": "technical", "context": "Python backend development"}'),
    ("Answer Evaluation",
     '{"question": "What is OOP?", "answer": "OOP stands for Object Oriented Programming. It has classes and objects."}'),
    ("Communication Assessment",
     '{"question": "Tell me about yourself", "answer": "Well, uh, I am a developer. I do coding and stuff. I like Python I guess."}')
]

print("="*60)
print("VERIFYING FINE-TUNED QWEN3-8B OUTPUTS")
print("="*60)
for name, prompt in TESTS:
    print(f"\nTask: {name}")
    print(f"Input: {prompt}")
    print(f"Response:\n{test_model(SYS, prompt)}")
    print("-"*60)

---
## Cell 8 — Export to GGUF (for Ollama Deployment)

In [ ]:
%%time
print("Quantizing and exporting model to GGUF (q4_k_m precision)...\n")

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print(f"\n[OK] Export complete! GGUF files saved to: {GGUF_DIR}")
for f in os.listdir(GGUF_DIR):
    if f.endswith(".gguf"):
        size_gb = os.path.getsize(os.path.join(GGUF_DIR, f)) / (1024**3)
        print(f"  -> {f} ({size_gb:.2f} GB)")

---
## Cell 9 — (Optional) Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, login

import os
HF_TOKEN = os.environ.get("HF_TOKEN", "")

try:
    if HF_TOKEN and HF_TOKEN.startswith("hf_"):
        if HF_TOKEN:
            login(token=HF_TOKEN)
        
        # Push LoRA adapter
        repo_id = "suren3101/skillnox-qwen3-8b-lora"
        print(f"Pushing LoRA adapter to {repo_id}...")
        model.push_to_hub(repo_id, token=HF_TOKEN)
        tokenizer.push_to_hub(repo_id, token=HF_TOKEN)
        print(f"[OK] LoRA adapter pushed to https://huggingface.co/{repo_id}")
        
        # Push GGUF
        gguf_repo = "suren3101/skillnox-qwen3-8b-gguf"
        api = HfApi()
        import os
        for f in os.listdir(GGUF_DIR):
            if f.endswith(".gguf"):
                print(f"Uploading {f} to {gguf_repo}...")
                api.upload_file(
                    path_or_fileobj=os.path.join(GGUF_DIR, f),
                    path_in_repo=f,
                    repo_id=gguf_repo,
                    repo_type="model",
                    token=HF_TOKEN if HF_TOKEN else None,
                )
                print(f"[OK] GGUF uploaded to https://huggingface.co/{gguf_repo}")
    else:
        print("[WARN] HF_TOKEN is empty or invalid. Skipping push.")
except Exception as e:
    print(f"[WARN] HuggingFace push failed: {e}")
